In [1]:
import os
import pickle
import numpy as np
import glob
from PIL import Image
from numba import njit
import time

### BaseLayer

In [2]:
class BaseLayer:
    def __init__(self, trainable=False):
        self.trainable = trainable

    def forward(self, X):
        raise NotImplementedError

    def backward(self, dY, cache):
        raise NotImplementedError

    def update(self, grad, lr):
        pass

    def state_dict(self):
        return {}

    def load_state_dict(self, state):
        pass

In [3]:
class Conv2DLayer(BaseLayer):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=0, backend='numpy'):
        super().__init__(trainable=True)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.backend = backend
        k = kernel_size
        self.weights = np.random.randn(out_channels, in_channels, k, k).astype(np.float32) * np.sqrt(2.0 / (in_channels * k * k))
        self.bias = np.zeros((out_channels,), dtype=np.float32)

    def forward(self, X):
        if self.backend == 'numba':
            y = conv2d_forward_nb(X, self.weights, self.bias, self.stride, self.padding)
        else:
            y = conv2d_forward_np(X, self.weights, self.bias, self.stride, self.padding)
        cache = (X, y)
        return y, cache

    def backward(self, dY, cache):
        X, _ = cache
        if self.backend == 'numba':
            dX, dW, db = conv2d_backward_nb(dY, X, self.weights, self.stride, self.padding)
        else:
            dX, dW, db = conv2d_backward_np(dY, X, self.weights, self.stride, self.padding)
        grad = {'weights': dW, 'bias': db}
        return dX, grad

    def update(self, grad, lr):
        self.weights -= lr * grad['weights']
        self.bias -= lr * grad['bias']

    def state_dict(self):
        return {
            'in_channels': self.in_channels,
            'out_channels': self.out_channels,
            'kernel_size': self.kernel_size,
            'stride': self.stride,
            'padding': self.padding,
            'weights': self.weights,
            'bias': self.bias,
            'backend': self.backend,
        }

    def load_state_dict(self, state):
        self.in_channels = state['in_channels']
        self.out_channels = state['out_channels']
        self.kernel_size = state['kernel_size']
        self.stride = state['stride']
        self.padding = state['padding']
        self.weights = state['weights']
        self.bias = state['bias']
        self.backend = state.get('backend', 'numpy')

In [4]:
def conv2d_forward_np(X, W, b, stride, padding):
    n, c, h, w = X.shape
    out_channels, _, kh, kw = W.shape
    h_pad = h + 2 * padding
    w_pad = w + 2 * padding
    xp = np.pad(X, ((0, 0), (0, 0), (padding, padding), (padding, padding)), mode='constant')
    h_out = (h_pad - kh) // stride + 1
    w_out = (w_pad - kw) // stride + 1
    out = np.zeros((n, out_channels, h_out, w_out), dtype=X.dtype)
    for i in range(h_out):
        for j in range(w_out):
            patch = xp[:, :, i*stride:i*stride+kh, j*stride:j*stride+kw]
            for oc in range(out_channels):
                out[:, oc, i, j] = np.sum(patch * W[oc:oc+1, :, :, :], axis=(1,2,3))
    out += b[None, :, None, None]
    return out


def conv2d_backward_np(dY, X, W, stride, padding):
    n, c, h, w = X.shape
    out_channels, _, kh, kw = W.shape
    h_pad = h + 2 * padding
    w_pad = w + 2 * padding
    xp = np.pad(X, ((0, 0), (0, 0), (padding, padding), (padding, padding)), mode='constant')
    dXp = np.zeros_like(xp)
    dW = np.zeros_like(W)
    db = np.zeros_like(W[:, 0, 0, 0])
    h_out, w_out = dY.shape[2], dY.shape[3]

    for i in range(h_out):
        for j in range(w_out):
            patch = xp[:, :, i*stride:i*stride+kh, j*stride:j*stride+kw]
            for oc in range(out_channels):
                dW[oc] += np.sum(patch * (dY[:, oc, i, j])[:, None, None, None], axis=0)
            for n_i in range(n):
                for oc in range(out_channels):
                    dXp[n_i, :, i*stride:i*stride+kh, j*stride:j*stride+kw] += W[oc] * dY[n_i, oc, i, j]

    for oc in range(out_channels):
        db[oc] = np.sum(dY[:, oc, :, :])

    if padding > 0:
        dX = dXp[:, :, padding:-padding, padding:-padding]
    else:
        dX = dXp
    return dX, dW / n, db / n

@njit
def conv2d_forward_nb(X, W, b, stride, padding):
        n, c, h, w = X.shape
        out_channels, _, kh, kw = W.shape
        h_pad = h + 2 * padding
        w_pad = w + 2 * padding
        h_out = (h_pad - kh) // stride + 1
        w_out = (w_pad - kw) // stride + 1
        out = np.zeros((n, out_channels, h_out, w_out), dtype=np.float32)

        xp = np.zeros((n, c, h_pad, w_pad), dtype=np.float32)
        for i in range(n):
            for j in range(c):
                for p in range(h):
                    for q in range(w):
                        xp[i, j, p+padding, q+padding] = X[i, j, p, q]

        for i in range(h_out):
            for j in range(w_out):
                for ni in range(n):
                    for oc in range(out_channels):
                        v = 0.0
                        for ci in range(c):
                            for ki in range(kh):
                                for kj in range(kw):
                                    v += xp[ni, ci, i*stride+ki, j*stride+kj] * W[oc, ci, ki, kj]
                        out[ni, oc, i, j] = v + b[oc]
        return out

@njit
def conv2d_backward_nb(dY, X, W, stride, padding):
        n, c, h, w = X.shape
        out_channels, _, kh, kw = W.shape
        h_pad = h + 2 * padding
        w_pad = w + 2 * padding
        xp = np.zeros((n, c, h_pad, w_pad), dtype=np.float32)
        for i in range(n):
            for j in range(c):
                for p in range(h):
                    for q in range(w):
                        xp[i, j, p+padding, q+padding] = X[i, j, p, q]

        dXp = np.zeros_like(xp)
        dW = np.zeros_like(W)
        db = np.zeros((out_channels,), dtype=np.float32)
        h_out = dY.shape[2]
        w_out = dY.shape[3]

        for i in range(h_out):
            for j in range(w_out):
                for ni in range(n):
                    for oc in range(out_channels):
                        for ci in range(c):
                            for ki in range(kh):
                                for kj in range(kw):
                                    dW[oc, ci, ki, kj] += xp[ni, ci, i*stride+ki, j*stride+kj] * dY[ni, oc, i, j]
                                    dXp[ni, ci, i*stride+ki, j*stride+kj] += W[oc, ci, ki, kj] * dY[ni, oc, i, j]
                        db[oc] += dY[ni, oc, i, j]

        dX = dXp[:, :, padding:padding+h, padding:padding+w] if padding > 0 else dXp
        return dX, dW / n, db / n


In [5]:
class ReLULayer(BaseLayer):
    def __init__(self, backend='numpy'):
        super().__init__(trainable=False)
        self.backend = backend

    def forward(self, X):
        y = np.maximum(0, X)
        cache = X
        return y, cache

    def backward(self, dY, cache):
        X = cache
        dX = dY * (X > 0)
        return dX, {}

In [6]:
class FlattenLayer(BaseLayer):
    def __init__(self):
        super().__init__(trainable=False)

    def forward(self, X):
        cache = X.shape
        return X.reshape(X.shape[0], -1), cache

    def backward(self, dY, cache):
        return dY.reshape(cache), {}

In [7]:
class LinearLayer(BaseLayer):
    def __init__(self, in_features, out_features, backend='numpy'):
        super().__init__(trainable=True)
        self.in_features = in_features
        self.out_features = out_features
        self.backend = backend
        self.weights = np.random.randn(out_features, in_features).astype(np.float32) * np.sqrt(2.0 / in_features)
        self.bias = np.zeros((out_features,), dtype=np.float32)

    def forward(self, X):
        y = X.dot(self.weights.T) + self.bias
        return y, X

    def backward(self, dY, cache):
        X = cache
        dW = dY.T.dot(X) / X.shape[0]
        db = np.mean(dY, axis=0)
        dX = dY.dot(self.weights)
        return dX, {'weights': dW, 'bias': db}

    def update(self, grad, lr):
        self.weights -= lr * grad['weights']
        self.bias -= lr * grad['bias']

    def state_dict(self):
        return {
            'in_features': self.in_features,
            'out_features': self.out_features,
            'weights': self.weights,
            'bias': self.bias,
            'backend': self.backend,
        }

    def load_state_dict(self, state):
        self.in_features = state['in_features']
        self.out_features = state['out_features']
        self.weights = state['weights']
        self.bias = state['bias']
        self.backend = state.get('backend', 'numpy')

In [8]:
class SoftmaxLayer(BaseLayer):
    def __init__(self):
        super().__init__(trainable=False)

    def forward(self, X):
        ex = np.exp(X - np.max(X, axis=1, keepdims=True))
        y = ex / np.sum(ex, axis=1, keepdims=True)
        return y, y

    def backward(self, dY, cache):
        y = cache
        dX = y * (dY - np.sum(dY * y, axis=1, keepdims=True))
        return dX, {}

In [9]:
class MaxPoolLayer(BaseLayer):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__(trainable=False)
        self.kernel_size = kernel_size
        self.stride = stride

    def forward(self, X):
        n, c, h, w = X.shape
        k = self.kernel_size
        s = self.stride
        h_out = (h - k) // s + 1
        w_out = (w - k) // s + 1
        y = np.zeros((n, c, h_out, w_out), dtype=X.dtype)
        mask = np.zeros_like(X, dtype=np.bool_)

        for i in range(h_out):
            for j in range(w_out):
                block = X[:, :, i*s:i*s+k, j*s:j*s+k]
                y[:, :, i, j] = np.max(block, axis=(2, 3))
                maxpos = (block == y[:, :, i, j][:, :, None, None])
                mask[:, :, i*s:i*s+k, j*s:j*s+k] = mask[:, :, i*s:i*s+k, j*s:j*s+k] | maxpos

        cache = (X.shape, mask)
        return y, cache

    def backward(self, dY, cache):
        input_shape, mask = cache
        dX = np.zeros(input_shape, dtype=dY.dtype)
        n, c, h_out, w_out = dY.shape
        k = self.kernel_size
        s = self.stride
        for i in range(h_out):
            for j in range(w_out):
                dX[:, :, i*s:i*s+k, j*s:j*s+k] += (mask[:, :, i*s:i*s+k, j*s:j*s+k] * dY[:, :, i:i+1, j:j+1])
        return dX, {}

### Loss

In [10]:
class BaseLossFunction:
    def calc(self, y_true, logits):
        raise NotImplementedError

    def derivative(self, y_true, logits):
        raise NotImplementedError


class CrossEntropyLoss(BaseLossFunction):
    def calc(self, y_true, logits):
        logits = logits.astype(np.float32)
        exp = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        probs = exp / np.sum(exp, axis=1, keepdims=True)
        n = y_true.shape[0]
        if y_true.ndim == 1:
            loss = -np.mean(np.log(probs[np.arange(n), y_true] + 1e-9))
        else:
            loss = -np.mean(np.sum(y_true * np.log(probs + 1e-9), axis=1))
        return loss

    def derivative(self, y_true, logits):
        logits = logits.astype(np.float32)
        exp = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        probs = exp / np.sum(exp, axis=1, keepdims=True)
        n = y_true.shape[0]
        if y_true.ndim == 1:
            grad = probs.copy()
            grad[np.arange(n), y_true] -= 1
            grad /= n
            return grad
        else:
            grad = (probs - y_true) / n
            return grad

### Optimazer

In [11]:
class BaseOptimizer:
    def __init__(self, loss_func):
        self.loss_func = loss_func

    def optimize(self, X, y, model):
        raise NotImplementedError


class GDOptimizer(BaseOptimizer):
    def __init__(self, loss_func, learning_rate=0.01):
        super().__init__(loss_func)
        self.learning_rate = learning_rate

    def optimize(self, X, y, model):
        logits, cache = model.predict(X)
        loss = self.loss_func.calc(y, logits)
        dL_dy = self.loss_func.derivative(y, logits)
        grads = model._get_grads(dL_dy, cache)
        model._update_weights(grads, self.learning_rate)
        return loss

### Model

In [12]:
class Model:
    def __init__(self, layers=None):
        self.layers = layers if layers is not None else []

    def predict(self, X):
        y = X
        cache = []
        for layer in self.layers:
            y, c = layer.forward(y)
            cache.append(c)
        return y, cache

    def _get_grads(self, dL_dy, cache):
        dy = dL_dy
        grads = []
        for layer, c in zip(self.layers[::-1], cache[::-1]):
            dy, grad = layer.backward(dy, c)
            grads.insert(0, grad)
        return grads

    def _update_weights(self, grads, learning_rate):
        for layer, grad in zip(self.layers, grads):
            if layer.trainable:
                layer.update(grad, learning_rate)

    def save_model(self, path):
        with open(path, 'wb') as f:
            pickle.dump(self.state_dict(), f)

    @classmethod
    def from_file(cls, path):
        with open(path, 'rb') as f:
            state = pickle.load(f)
        model = cls()
        model.load_state_dict(state)
        return model

    def state_dict(self):
        return {'layers': [(layer.__class__.__name__, layer.state_dict()) for layer in self.layers]}

    def load_state_dict(self, state):
        self.layers = []
        for name, layer_state in state['layers']:
            if name == 'Conv2DLayer':
                layer = Conv2DLayer(1, 1)
            elif name == 'ReLULayer':
                layer = ReLULayer()
            elif name == 'FlattenLayer':
                layer = FlattenLayer()
            elif name == 'LinearLayer':
                layer = LinearLayer(1, 1)
            elif name == 'SoftmaxLayer':
                layer = SoftmaxLayer()
            elif name == 'MaxPoolLayer':
                layer = MaxPoolLayer()
            layer.load_state_dict(layer_state)
            self.layers.append(layer)

In [13]:
def simple_cnn(image_size=8, backend='numpy'):
    in_flat = (image_size // 2) * (image_size // 2) * 4
    return Model(layers=[
        Conv2DLayer(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1, backend=backend),
        ReLULayer(backend=backend),
        MaxPoolLayer(kernel_size=2, stride=2),
        FlattenLayer(),
        LinearLayer(in_features=in_flat, out_features=16, backend=backend),
        ReLULayer(backend=backend),
        LinearLayer(in_features=16, out_features=2, backend=backend),
        SoftmaxLayer(),
    ])

загрузка данных

In [14]:
def load_data(base_dir, image_size=8):
    X = []
    y = []
    for label, subdir in enumerate(['Cat', 'Dog']):
        folder = os.path.join(base_dir, subdir)
        for filename in sorted(glob.glob(os.path.join(folder, '*.jpg'))):
            try:
                im = Image.open(filename).convert('L').resize((image_size, image_size))
                arr = np.asarray(im, dtype=np.float32) / 255.0
                X.append(arr)
                y.append(label)
            except Exception as e:
                print('skip', filename, e)
    X = np.stack(X, axis=0)
    X = X[:, None, :, :]
    y = np.array(y, dtype=np.int64)
    perm = np.random.permutation(len(y))
    X = X[perm]
    y = y[perm]
    return X, y


обучение

In [15]:
def train_model(base_dir, epochs=40, lr=0.1, image_size=8, backend='numpy'):
    np.random.seed(42)
    X, y = load_data(base_dir, image_size=image_size)
    n = len(y)
    split = int(n * 0.9)
    X_train, y_train = X[:split], y[:split]
    X_val, y_val = X[split:], y[split:]

    model = simple_cnn(image_size=image_size, backend=backend)
    loss_fn = CrossEntropyLoss()

    epoch_times = []
    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()

        logits, cache = model.predict(X_train)
        loss = loss_fn.calc(y_train, logits)
        preds = np.argmax(logits, axis=1)
        acc = np.mean(preds == y_train)

        dL_dy = loss_fn.derivative(y_train, logits)
        grads = model._get_grads(dL_dy, cache)
        model._update_weights(grads, lr)

        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)
        train_losses.append(loss)
        train_accs.append(acc)

        val_logits, _ = model.predict(X_val)
        val_loss = loss_fn.calc(y_val, val_logits)
        val_acc = np.mean(np.argmax(val_logits, axis=1) == y_val)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(f"epoch {epoch:02d}    t {epoch_time:.3f}s    train_loss {loss:.4f}   val_loss {val_loss:.4f}")

    history = {
        'epoch_times': np.array(epoch_times),
        'train_losses': np.array(train_losses),
        'train_accs': np.array(train_accs),
        'val_losses': np.array(val_losses),
        'val_accs': np.array(val_accs),
    }

    return model, (X_val, y_val), history

In [16]:
def evaluate_model(model, X, y):
    logits, _ = model.predict(X)
    preds = np.argmax(logits, axis=1)
    acc = np.mean(preds == y)
    loss = CrossEntropyLoss().calc(y, logits)
    print(f"Evaluation: loss={loss:.4f}, acc={acc:.4f}")
    return loss, acc


def benchmark_backend(base_dir, backend, epochs=40, lr=0.2, image_size=8):
    print(f"=== {backend} ===")
    start = time.time()
    model, (X_val, y_val), history = train_model(base_dir, epochs=epochs, lr=lr, image_size=image_size, backend=backend)
    dt = time.time() - start
    fname = f"cnn_{backend}.pkl"
    model.save_model(fname)
    print(f"Saved model to {fname}")
    loss, acc = evaluate_model(model, X_val, y_val)
    return dt, loss, acc, history

In [17]:
if __name__ == '__main__':
    base_dir = os.path.dirname('C:/Users/user/Desktop/cuda/')
    results = []
    histories = {}
    cpu_dt, cpu_loss, cpu_acc, cpu_history = benchmark_backend(base_dir, backend='numpy', epochs=15, lr=0.1, image_size=8)
    results.append(('numpy', cpu_dt, cpu_loss, cpu_acc))
    histories['numpy'] = cpu_history

    numba_dt, numba_loss, numba_acc, numba_history = benchmark_backend(base_dir, backend='numba', epochs=15, lr=0.1, image_size=8)
    results.append(('numba', numba_dt, numba_loss, numba_acc))
    histories['numba'] = numba_history

=== numpy ===
epoch 01    t 0.187s    train_loss 0.7053   val_loss 0.6902
epoch 02    t 0.187s    train_loss 0.7052   val_loss 0.6903
epoch 03    t 0.190s    train_loss 0.7052   val_loss 0.6903
epoch 04    t 0.183s    train_loss 0.7052   val_loss 0.6903
epoch 05    t 0.187s    train_loss 0.7051   val_loss 0.6904
epoch 06    t 0.185s    train_loss 0.7051   val_loss 0.6904
epoch 07    t 0.185s    train_loss 0.7050   val_loss 0.6904
epoch 08    t 0.183s    train_loss 0.7050   val_loss 0.6904
epoch 09    t 0.188s    train_loss 0.7050   val_loss 0.6905
epoch 10    t 0.190s    train_loss 0.7049   val_loss 0.6905
epoch 11    t 0.190s    train_loss 0.7049   val_loss 0.6905
epoch 12    t 0.188s    train_loss 0.7048   val_loss 0.6906
epoch 13    t 0.190s    train_loss 0.7048   val_loss 0.6906
epoch 14    t 0.188s    train_loss 0.7048   val_loss 0.6906
epoch 15    t 0.209s    train_loss 0.7047   val_loss 0.6906
Saved model to cnn_numpy.pkl
Evaluation: loss=0.6906, acc=0.4500
=== numba ===
epoch 0